# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [12]:

#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier


from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [9]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6606 entries, 0 to 8692
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        6606 non-null   float64
 1   RoomService                6606 non-null   float64
 2   FoodCourt                  6606 non-null   float64
 3   ShoppingMall               6606 non-null   float64
 4   Spa                        6606 non-null   float64
 5   VRDeck                     6606 non-null   float64
 6   HomePlanet_Europa          6606 non-null   bool   
 7   HomePlanet_Mars            6606 non-null   bool   
 8   CryoSleep_True             6606 non-null   bool   
 9   Cabin_B                    6606 non-null   bool   
 10  Cabin_C                    6606 non-null   bool   
 11  Cabin_D                    6606 non-null   bool   
 12  Cabin_E                    6606 non-null   bool   
 13  Cabin_F                    6606 non-null   bool   
 1

In [10]:
spaceship.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000
mean,28.894036,222.991674,478.958523,178.356494,313.161520,303.780048
std,14.533429,644.987936,1678.592291,576.328407,1144.016291,1127.142166
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,82.750000,30.000000,65.000000,52.000000
max,79.000000,9920.000000,29813.000000,12253.000000,22408.000000,20336.000000


In [8]:
spaceship = spaceship.dropna()
spaceship.shape

(6606, 20)

In [ ]:

# Extract the first letter of the 'Cabin' column to represent the deck
spaceship['Cabin'] = spaceship['Cabin'].str[0]

spaceship.drop(columns=['PassengerId', 'Name',], inplace=True)

spaceship = pd.get_dummies(spaceship, columns=['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP','Transported'], drop_first=True)

**Perform Train Test Split**

In [6]:
# Separate features and target variable
X = spaceship.drop(columns=['Transported_True'])
y = spaceship['Transported_True']

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

In [14]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns = X_train.columns)
X_train_scaled.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,-0.267707,-0.350087,0.694479,-0.302537,0.300959,0.256747,1.727691,-0.511566,-0.737649,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,-0.654595,-0.019459,-0.324002,-1.495247,-0.156295
1,0.492810,-0.350087,1.252114,2.468142,2.231549,-0.263876,1.727691,-0.511566,-0.737649,-0.320782,3.234508,-0.244474,-0.338774,-0.698892,-0.654595,-0.019459,-0.324002,0.668786,-0.156295
2,0.354535,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,1.954781,1.355658,-0.320782,-0.309166,-0.244474,-0.338774,1.430835,-0.654595,-0.019459,-0.324002,-1.495247,-0.156295
3,0.561948,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,-0.511566,1.355658,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,-0.324002,-1.495247,-0.156295
4,-0.475120,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,-0.511566,1.355658,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,3.086405,-1.495247,-0.156295


In [17]:
X_test_scaled = pd.DataFrame(X_test_scaled, columns = X_test.columns)
X_test_scaled.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.008845,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,1.727691,-0.511566,1.355658,3.117379,-0.309166,-0.244474,-0.338774,-0.698892,-0.654595,-0.019459,-0.324002,0.668786,-0.156295
1,-1.097362,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,-0.511566,1.355658,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,-0.324002,-1.495247,-0.156295
2,1.460741,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,-0.511566,1.355658,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,-0.324002,0.668786,-0.156295
3,-1.581327,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,-0.578807,-0.511566,-0.737649,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,-0.324002,-1.495247,-0.156295
4,0.561948,-0.022472,-0.284754,0.702399,-0.272758,-0.206786,-0.578807,-0.511566,-0.737649,-0.320782,-0.309166,-0.244474,-0.338774,-0.698892,1.527663,-0.019459,-0.324002,0.668786,-0.156295


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [18]:
# Bagging
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=6), n_estimators=100, max_samples=100)
bagging_clf.fit(X_train_scaled, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=6),
                  max_samples=100, n_estimators=100)

In [19]:
pred_bagg = bagging_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_bagg))
print("Precision:", precision_score(y_test, pred_bagg, average='weighted'))
print("Recall:", recall_score(y_test, pred_bagg, average='weighted'))
print("F1 Score:", f1_score(y_test, pred_bagg, average='weighted'))

Accuracy: 0.7934947049924357
Precision: 0.7948099598286302
Recall: 0.7934947049924357
F1 Score: 0.7931440386401263


In [20]:
pasting_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=6), n_estimators=100, max_samples=100)
pasting_clf.fit(X_train_scaled, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=6),
                  max_samples=100, n_estimators=100)

In [21]:
pred_past = pasting_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_past))
print("Precision:", precision_score(y_test, pred_past, average='weighted'))
print("Recall:", recall_score(y_test, pred_past, average='weighted'))
print("F1 Score:", f1_score(y_test, pred_past, average='weighted'))

Accuracy: 0.8086232980332829
Precision: 0.8086489932745825
Recall: 0.8086232980332829
F1 Score: 0.8085986510362767


- Random Forests

In [27]:
forest = RandomForestClassifier(n_estimators=100, max_depth=30)
forest.fit(X_train_scaled, y_train)

RandomForestClassifier(max_depth=30)

In [28]:
pred_for = forest.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_for))
print("Precision:", precision_score(y_test, pred_for, average='weighted'))
print("Recall:", recall_score(y_test, pred_for, average='weighted'))
print("F1 Score:", f1_score(y_test, pred_for, average='weighted'))

Accuracy: 0.8048411497730711
Precision: 0.8051315923272505
Recall: 0.8048411497730711
F1 Score: 0.8048371297473181


- Gradient Boosting

In [34]:
gb_clf = GradientBoostingClassifier(max_depth=10, n_estimators=100)
gb_clf.fit(X_train_scaled, y_train)

GradientBoostingClassifier(max_depth=10)

In [35]:

pred_gb = gb_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_gb))
print("Precision:", precision_score(y_test, pred_gb, average='weighted'))
print("Recall:", recall_score(y_test, pred_gb, average='weighted'))
print("F1 Score:", f1_score(y_test, pred_gb, average='weighted'))

Accuracy: 0.81089258698941
Precision: 0.8109023591009636
Recall: 0.81089258698941
F1 Score: 0.8108956170574643


- Adaptive Boosting

In [38]:
base_estimator = DecisionTreeClassifier(max_depth=10)
ab_clf = AdaBoostClassifier(estimator=base_estimator, n_estimators=100)
ab_clf.fit(X_train_scaled, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=10),
                   n_estimators=100)

In [39]:
pred_ab = ab_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_ab))
print("Precision:", precision_score(y_test, pred_ab, average='weighted'))
print("Recall:", recall_score(y_test, pred_ab, average='weighted'))
print("F1 Score:", f1_score(y_test, pred_ab, average='weighted'))

Accuracy: 0.794251134644478
Precision: 0.7943243744292446
Recall: 0.794251134644478
F1 Score: 0.794258669421708


Which model is the best and why?

In [44]:
# Compare the main metrics for each model
results = {
    "Bagging": {
        "Accuracy": accuracy_score(y_test, pred_bagg),
        "Precision": precision_score(y_test, pred_bagg, average='weighted'),
        "Recall": recall_score(y_test, pred_bagg, average='weighted'),
        "F1": f1_score(y_test, pred_bagg, average='weighted')
    },
    "Pasting": {
        "Accuracy": accuracy_score(y_test, pred_past),
        "Precision": precision_score(y_test, pred_past, average='weighted'),
        "Recall": recall_score(y_test, pred_past, average='weighted'),
        "F1": f1_score(y_test, pred_past, average='weighted')
    },
    "Random Forest": {
        "Accuracy": accuracy_score(y_test, pred_for),
        "Precision": precision_score(y_test, pred_for, average='weighted'),
        "Recall": recall_score(y_test, pred_for, average='weighted'),
        "F1": f1_score(y_test, pred_for, average='weighted')
    },
    "Gradient Boosting": {
        "Accuracy": accuracy_score(y_test, pred_gb),
        "Precision": precision_score(y_test, pred_gb, average='weighted'),
        "Recall": recall_score(y_test, pred_gb, average='weighted'),
        "F1": f1_score(y_test, pred_gb, average='weighted')
    },
    "AdaBoost": {
        "Accuracy": accuracy_score(y_test, pred_ab),
        "Precision": precision_score(y_test, pred_ab, average='weighted'),
        "Recall": recall_score(y_test, pred_ab, average='weighted'),
        "F1": f1_score(y_test, pred_ab, average='weighted')
    }
}

results_df = pd.DataFrame(results).T
display(results_df)

best_model = results_df['F1'].idxmax()


,Accuracy,Precision,Recall,F1
Bagging,0.793495,0.794810,0.793495,0.793144
Pasting,0.808623,0.808649,0.808623,0.808599
Random Forest,0.804841,0.805132,0.804841,0.804837
Gradient Boosting,0.810893,0.810902,0.810893,0.810896
AdaBoost,0.794251,0.794324,0.794251,0.794259


In [ ]:
"""
The best model is: Gradient Boosting (based on F1 Score).
This model is preferred because it achieves the highest balance between precision and recall, which is important for classification tasks.

"""

'\nThe best model is: Gradient Boosting (based on F1 Score).\nThis model is preferred because it achieves the highest balance between precision and recall, which is important for classification tasks.\n\n'